## Task 1: Fit PCA and Plot Explained Variance

First, we need to load the Mall Customers dataset. Please ensure the `Mall_Customers.csv` file is uploaded to your Colab environment or accessible from a mounted drive. Then, we will standardize the features, apply PCA, and analyze the explained variance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load the dataset
# Assuming 'Mall_Customers.csv' is in the current directory or '/content/sample_data/'
try:
    df = pd.read_csv('Mall_Customers.csv')
except FileNotFoundError:
    print("Mall_Customers.csv not found. Please upload it or adjust the path.")
    # As a fallback for demonstration, you might load a sample if available
    # For this task, we will assume it's available.
    # raise

print("Dataset loaded successfully. Displaying the first 5 rows:")
display(df.head())

### Step 1: Standardize the features

We will select numerical features from the dataset (`Age`, `Annual Income (k$)`, `Spending Score (1-100)`) and standardize them using `StandardScaler`. This is crucial for PCA as it is sensitive to the scale of the features.

In [ ]:
# Select relevant numerical features for PCA
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features standardized successfully. First 5 rows of scaled data:")
print(X_scaled[:5])

### Step 2 & 3: Fit PCA with all components and Plot Explained Variance

Now, we'll apply PCA with `n_components=None` to capture all principal components and then plot the `explained_variance_ratio_` to create a scree plot. This plot helps us understand how much variance each component accounts for.

In [ ]:
# Fit PCA with all components
pca = PCA(n_components=None)
pca.fit(X_scaled)

# Plot explained_variance_ratio_ as a bar chart (scree plot)
plt.figure(figsize=(10, 6))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot: Explained Variance Ratio per Principal Component')
plt.xticks(range(1, len(pca.explained_variance_ratio_) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Step 4: Answer: How many components capture 90% of variance? 95%?

Let's calculate the cumulative explained variance to determine the number of components needed to capture 90% and 95% of the total variance.

In [ ]:
# Calculate cumulative explained variance
cumsum_variance = np.cumsum(pca.explained_variance_ratio_)

# Find number of components for 90% variance
components_90_percent = np.where(cumsum_variance >= 0.90)[0][0] + 1

# Find number of components for 95% variance
components_95_percent = np.where(cumsum_variance >= 0.95)[0][0] + 1

print(f"Cumulative Explained Variance: {np.round(cumsum_variance, 4)}")
print(f"Number of components to capture 90% of variance: {components_90_percent}")
print(f"Number of components to capture 95% of variance: {components_95_percent}")

# Plot cumulative explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumsum_variance) + 1), cumsum_variance, marker='o', linestyle='--')
plt.axhline(y=0.90, color='r', linestyle='-', label='90% Variance')
plt.axvline(x=components_90_percent, color='r', linestyle='--', label=f'{components_90_percent} components for 90%')
plt.axhline(y=0.95, color='g', linestyle='-', label='95% Variance')
plt.axvline(x=components_95_percent, color='g', linestyle='--', label=f'{components_95_percent} components for 95%')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance vs. Number of Components')
plt.xticks(range(1, len(cumsum_variance) + 1))
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

## Task 2: Reduce to 2D and Visualize

Now we will transform the scaled data to its first two principal components (PC1 and PC2) and visualize these components. Since we don't have explicit cluster labels from 'Activity 1' in this notebook, we'll initially plot without them, and then I'll show how to incorporate them if they were available.

In [ ]:
# Step 1: Fit PCA with n_components=2 and transform the data
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

print("Data transformed to 2 principal components. First 5 rows:")
print(X_pca[:5])

# Step 2: Create a scatter plot of PC1 vs PC2
plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=50, alpha=0.8, c='steelblue', edgecolor='k')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.title('2D PCA Projection of Mall Customer Data')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\nObservation:\nThe scatter plot shows how the data points are distributed in the reduced 2D PCA space. We can observe some groupings, but without cluster labels from previous analysis, it's hard to definitively say if clusters separate cleanly from this plot alone. If we had `cluster_labels` from K-Means or DBSCAN, we could color these points accordingly.")

## Task 3: Interpret the Principal Components

To understand what each principal component represents, we need to examine the PCA loadings (`pca.components_`). These loadings indicate how much each original feature contributes to each principal component.

In [ ]:
# Step 1: Print pca.components_ as a DataFrame
# Use the 'features' list defined earlier for column names

# Re-fit PCA with 2 components for precise loadings of the chosen dimensions
# (though the loadings are extracted from the original pca object which had all components)
# For clarity, let's use pca_2d which was explicitly fitted for 2 components

pca_loadings_df = pd.DataFrame(
    pca_2d.components_,
    columns=features,
    index=['PC1', 'PC2']
)

print("PCA Loadings (Components):")
display(pca_loadings_df)

# Step 2 & 3: Identify contributing features and give interpretable names
print("\nInterpretation:\n")

# PC1 Analysis
print("PC1 (Principal Component 1):")
print(f"  Largest absolute loading is for '{pca_loadings_df.loc['PC1'].abs().idxmax()}' ({pca_loadings_df.loc['PC1'].max():.2f})")
print(f"  Second largest absolute loading is for '{pca_loadings_df.loc['PC1'].abs().nlargest(2).index[-1]}' ({pca_loadings_df.loc['PC1'].nlargest(2).values[-1]:.2f})")
print("  PC1 seems to primarily represent 'Annual Income' and 'Spending Score'. Given their positive correlation in this component, it could be interpreted as 'Economic Activity' or 'Customer Engagement/Spending Power'.")

# PC2 Analysis
print("\nPC2 (Principal Component 2):")
print(f"  Largest absolute loading is for '{pca_loadings_df.loc['PC2'].abs().idxmax()}' ({pca_loadings_df.loc['PC2'].max():.2f})")
print("  PC2 has a strong loading for 'Age'. This component primarily differentiates customers by their age.")
print("  PC2 can be interpreted as 'Age Profile' or 'Generational Difference'.")